# 01 Basics: setup, lattices and beams

This notebook runs the seven scripts of `01_basics/` in order with `%run`, so it runs exactly the same code as the command line (`python 01_basics/NN_name.py`). The text here adds the explanation; each script's docstring says what it shows, what it needs and what it checks. Every script ends with physics checks (`assert`s): a cell that fails means the check failed.

Nothing here needs Docker: these are all tier A. Outputs go to `../output/<script name>/`.

## 01 Environment check

Which versions of SIMBA, LAURA and laura-lattices are installed and where from, which Python tracking codes can be imported, and whether a container runtime is there for the compiled codes. Missing codes are reported, not failures: they only decide which examples you can run.

In [ ]:
%run 01_environment_check.py

## 02 Build a lattice in Python

A lattice in LAURA is a set of **elements** (each with a `machine_area`), **sections** (ordered lists of element names) and **layouts** (ordered lists of sections, a beam path). Positions are the element's middle, `physical.middle.z`, in m; quadrupole strengths are integrated, `k1l = k1 * length` in 1/m.

The script builds one FODO cell, writes it as LAURA YAML (the layout the machines in laura-lattices use) and reads it back.

In [ ]:
%run 02_build_lattice_python.py

## 03 Load a real machine

`Framework(master_lattice=...)` points SIMBA at a machine folder; `loadSettings(".def")` reads a lattice definition file, which splits the beam path into **lines**, each run by one code. Here: JFEL, an example S-band FEL linac from laura-lattices.

Things that catch people out:
- `framework["NAME"]` is a line if NAME is a line, otherwise an element.
- `element.length` is ambiguous; use `element.physical.length`. Strengths like `element.k1l` read directly.
- `LAURA.elements_between(end, start)` takes the end first; use keywords.

The check compares SIMBA's line with LAURA's layout. Downstream of the chicane they differ by the chicane's extra path length (27.8 mm), because the JFEL section files give s along the straight line there: see `ISSUES.md`.

In [ ]:
%run 03_load_machine.py

## 04 The `.def` file

Open `jfel_example.def` alongside: it is a commented `.def` with two lines in two codes, a starting beam file, a Twiss rematch, a chicane group, a generator and inline `changes:`. Relative paths in a `.def` are relative to the master lattice, and `$master_lattice$` in a value is replaced by its folder.

SIMBA ignores top-level keys it does not know, without a warning: a misspelt `chnages:` does nothing.

In [ ]:
print(open('jfel_example.def').read())

In [ ]:
%run 04_the_def_file.py

## 05 Generate a beam

SIMBA's Python generator takes sizes in m, momenta in **eV/c**, charge in C (positive; the sign comes from the species) and correlations as coefficients. It does not take Twiss parameters, so the script converts them:

$\sigma_x = \sqrt{\varepsilon\beta}$, $\sigma_{p_x} = p\sqrt{\varepsilon\gamma}$, correlation $= -\alpha/\sqrt{1+\alpha^2}$.

Two things change what you get from what you asked for:
- each Gaussian is cut at 3σ by default, which trims the rms size by a factor 0.9866 and the emittance by 0.9733 (the Twiss parameters are unchanged);
- for a flat-top, `sigma_z` sets the half-width (±3 `sigma_z`), not the rms.

There is no seed argument: call `np.random.seed()` first.

In [ ]:
%run 05_generate_beam.py

## 06 Beam files

`rbf.load_file(path)` reads any format SIMBA knows, choosing the reader from the file name. Each `write_*_beam_file` writes one format. Time-based formats (elegant SDDS, Ocelot) do not store the bunch's absolute z, and a GPT initial distribution (GDF) has no t.

Two known bugs show as XFAIL: an SDDS file reads back with the wrong sign of charge, and `write_HDF5_beam_file` fails for a beam not read from a code's output.

In [ ]:
%run 06_beam_io_and_conversion.py

## 07 Analysis and plots

Beam properties live under `beam.Particles` (`sigmas`, `emittance`, `twiss`, `slice`); every value is a `UnitValue` in SI units (`.val` for plain numpy). Note that `sigmas.momentum_spread` is in eV/c, not relative.

The plotting functions draw into the current matplotlib figure; the script saves them to files, shown below.

In [ ]:
%run 07_beam_analysis_and_plots.py

In [ ]:
from IPython.display import Image, display
for name in ["screen_xy", "t_cpz", "slices"]:
    display(Image(filename=f"../output/07_beam_analysis_and_plots/{name}.png"))